# Student Performance Analytics

## Complete Source Code

This notebook implements the Student Performance Analytics project using the UCI Student Performance dataset. It performs data loading, cleaning, exploratory analysis, preprocessing, performance categorization, Random Forest classification, evaluation, prediction, and dashboard-ready analytics.

**Dataset:** UCI Student Performance, Portuguese-language dataset (`student-por.csv`), 649 records.

**Primary model:** Random Forest Classifier.


In [ ]:
# Install required packages if needed
# Run this cell once in a new environment.
# %pip install pandas numpy matplotlib seaborn scikit-learn ucimlrepo


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


## 2. Load the Dataset


In [ ]:
# Preferred method: download directly from the UCI Machine Learning Repository.
from ucimlrepo import fetch_ucirepo

student_performance = fetch_ucirepo(id=320)
X_all = student_performance.data.features.copy()
y_all = student_performance.data.targets.copy()

# The UCI package exposes the common attributes and target separately.
# We use the Portuguese-language dataset because it contains 649 records.
DATA_URL = 'https://archive.ics.uci.edu/dataset/320/student'

# If a local student-por.csv is already available, the following alternative can be used:
# df = pd.read_csv('student-por.csv', sep=';')

print('Feature shape:', X_all.shape)
print('Target shape:', y_all.shape)
display(X_all.head())


In [ ]:
# Build one working DataFrame from the UCI features and target.
df = X_all.copy()

if 'G3' in y_all.columns:
    df['G3'] = y_all['G3']
else:
    # Fallback for versions of the package where G3 is already included in features.
    if 'G3' not in df.columns:
        raise ValueError('G3 target column was not found. Check the UCI package version.')

print('Dataset shape:', df.shape)
print('Number of records:', len(df))
print('Number of columns:', len(df.columns))
display(df.head())


## 3. Data Understanding and Cleaning


In [ ]:
print('Data types:')
display(df.dtypes)

print('Missing values:')
display(df.isnull().sum().sort_values(ascending=False).head(10))

print('Duplicate rows:', df.duplicated().sum())

# Remove exact duplicate records if present.
df = df.drop_duplicates().reset_index(drop=True)

# Ensure grade columns are numeric.
for col in ['G1', 'G2', 'G3']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['G3']).reset_index(drop=True)
print('Shape after cleaning:', df.shape)


## 4. Create Student Performance Categories


In [ ]:
def performance_category(grade):
    """Convert final grade (0-20) into a performance category."""
    if grade < 10:
        return 'Low'
    elif grade < 15:
        return 'Average'
    return 'High'

df['Performance_Category'] = df['G3'].apply(performance_category)

print(df['Performance_Category'].value_counts())
display(df[['G3', 'Performance_Category']].head(10))


## 5. Exploratory Data Analysis (EDA)


In [ ]:
print(df[['G1', 'G2', 'G3', 'absences', 'studytime', 'failures']].describe())


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Performance_Category', order=['Low', 'Average', 'High'])
plt.title('Student Performance Category Distribution')
plt.xlabel('Performance Category')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='absences', y='G3', hue='Performance_Category')
plt.title('Absences vs Final Grade')
plt.xlabel('Number of Absences')
plt.ylabel('Final Grade (G3)')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Performance_Category', y='studytime', order=['Low', 'Average', 'High'])
plt.title('Study Time by Performance Category')
plt.xlabel('Performance Category')
plt.ylabel('Study Time Level')
plt.tight_layout()
plt.show()


## 6. Feature Selection and Target Preparation


In [ ]:
# G3 is the target and is not used as an input feature.
# G1 and G2 are retained because the UCI documentation identifies them as prior-period grades.
target = 'Performance_Category'
drop_columns = ['G3', target]

X = df.drop(columns=drop_columns)
y = df[target]

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

print('Categorical features:', categorical_features)
print('Numeric features:', numeric_features)


## 7. Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training records:', len(X_train))
print('Testing records:', len(X_test))


## 8. Data Preprocessing Pipeline


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numeric', 'passthrough', numeric_features)
    ]
)


## 9. Train the Random Forest Classifier


In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

pipeline.fit(X_train, y_train)
print('Random Forest model training completed successfully.')


## 10. Predict Student Performance


In [ ]:
y_pred = pipeline.predict(X_test)

prediction_results = X_test.copy()
prediction_results['Actual_Performance'] = y_test.values
prediction_results['Predicted_Performance'] = y_pred

display(prediction_results[['Actual_Performance', 'Predicted_Performance']].head(15))


## 11. Model Evaluation


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, labels=['Low', 'Average', 'High'], zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['Low', 'Average', 'High'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low', 'Average', 'High'])
disp.plot()
plt.title('Confusion Matrix - Random Forest')
plt.tight_layout()
plt.show()


## 12. Feature Importance


In [ ]:
# Obtain transformed feature names and Random Forest importances.
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = pipeline.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).head(15)

display(importance_df)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x='Importance', y='Feature')
plt.title('Top 15 Feature Importances')
plt.tight_layout()
plt.show()


## 13. Dashboard-Ready Analytics


In [ ]:
dashboard_summary = {
    'Total Students': len(df),
    'Average Final Grade': round(df['G3'].mean(), 2),
    'Average Absences': round(df['absences'].mean(), 2),
    'High Performers': int((df['Performance_Category'] == 'High').sum()),
    'Average Performers': int((df['Performance_Category'] == 'Average').sum()),
    'Low Performers': int((df['Performance_Category'] == 'Low').sum())
}

display(pd.DataFrame([dashboard_summary]))

category_summary = df.groupby('Performance_Category').agg(
    Students=('Performance_Category', 'size'),
    Average_Grade=('G3', 'mean'),
    Average_Absences=('absences', 'mean'),
    Average_Study_Time=('studytime', 'mean')
).reindex(['Low', 'Average', 'High']).round(2)

display(category_summary)


## 14. Example Prediction for a Student Record


In [ ]:
# Example: use one test record as an example of an unseen student record.
example_student = X_test.iloc[[0]]
example_prediction = pipeline.predict(example_student)[0]

print('Predicted performance category:', example_prediction)
display(example_student)


## 15. Conclusion

The completed workflow loads the UCI Student Performance dataset, cleans and analyzes the data, creates performance categories, preprocesses mixed data types, trains a Random Forest Classifier, evaluates predictions, and produces analytics suitable for an educational dashboard. The same pipeline can be extended with a Streamlit or Power BI dashboard for interactive reporting.

### Dataset Reference
UCI Machine Learning Repository – Student Performance: https://archive.ics.uci.edu/dataset/320/student
